> **MIT License** - Copyright (c) 2026 QuantWall. Full text: https://opensource.org/license/mit
>
> **Disclaimer.** This content is general information and software-engineering education only. It is not financial product advice, a recommendation, or an offer to buy or sell any financial product, and it does not take into account your objectives, financial situation or needs. The author does not hold an Australian Financial Services Licence and is not a registered Commodity Trading Advisor. Trading futures and leveraged products carries a high risk of loss. Consult a licensed adviser before making any financial decision.
>
> **HYPOTHETICAL OR SIMULATED PERFORMANCE RESULTS HAVE CERTAIN LIMITATIONS. UNLIKE AN ACTUAL PERFORMANCE RECORD, SIMULATED RESULTS DO NOT REPRESENT ACTUAL TRADING. ALSO, SINCE THE TRADES HAVE NOT BEEN EXECUTED, THE RESULTS MAY HAVE UNDER-OR-OVER COMPENSATED FOR THE IMPACT, IF ANY, OF CERTAIN MARKET FACTORS, SUCH AS LACK OF LIQUIDITY. SIMULATED TRADING PROGRAMS IN GENERAL ARE ALSO SUBJECT TO THE FACT THAT THEY ARE DESIGNED WITH THE BENEFIT OF HINDSIGHT. NO REPRESENTATION IS BEING MADE THAT ANY ACCOUNT WILL OR IS LIKELY TO ACHIEVE PROFIT OR LOSSES SIMILAR TO THOSE SHOWN.**
>
> All data shown is synthetic or publicly available sample data. Features are standard textbook baselines (lagged returns, ATR, RSI, moving averages, volume). No proprietary signal is disclosed or implied.

# What a backtest is: the kitchen-table test, in code

Every price here is made up (a random walk with no edge). The rule: after two down days in a row (close below open), buy at the next open and sell at that day's close. The code below is the same as the animations' (`animations_manim/s1_sim.py`).

In [ ]:
import numpy as np

import numpy as np


def bars(n=500, seed=0, sd=0.01, start=100.0):
    """Daily open and close. The open is yesterday's close plus an overnight move; the close adds the day's move."""
    rng = np.random.default_rng(seed)
    over = rng.normal(0, sd * 0.4, n)
    day = rng.normal(0, sd, n)
    o, c = np.empty(n), np.empty(n)
    prev = start
    for i in range(n):
        o[i] = prev * (1 + over[i])
        c[i] = o[i] * (1 + day[i])
        prev = c[i]
    return o, c


def two_down_rule(o, c):
    """Honest version: decide after day t closes (two down days), trade day t+1 from its open to its close."""
    down = c < o
    sig = np.zeros(len(o), bool)
    sig[2:] = down[:-2] & down[1:-1]           # days t-1 and t-2 were down: trade day t
    ret = np.where(sig, c / o - 1, 0.0)
    return sig, ret


def peeking_rule(o, c):
    """The hook: 'buy at the open on days that close up'. It needs the close, which is still under the paper."""
    sig = c > o
    return sig, np.where(sig, c / o - 1, 0.0)


def summary(sig, ret):
    r = ret[sig]
    return {"trades": int(sig.sum()), "wins": int((r > 0).sum()), "win_share": float((r > 0).mean()) if len(r) else 0.0,
            "total": float(np.prod(1 + r) - 1), "worst": float(r.min()) if len(r) else 0.0}


# ---------------------------------------------------------------------------------------------
# The hook: a perfect backtest (two years of synthetic days)
# ---------------------------------------------------------------------------------------------
def hook(seed=0):
    o, c = bars(500, seed)
    p_sig, p_ret = peeking_rule(o, c)
    # the honest twin: decide at the open with what is above the paper (yesterday closed up), same trade
    h_sig = np.zeros(len(o), bool)
    h_sig[1:] = (c > o)[:-1]
    h_ret = np.where(h_sig, c / o - 1, 0.0)
    return {"peek": summary(p_sig, p_ret), "honest": summary(h_sig, h_ret),
            "peek_curve": np.cumprod(1 + p_ret), "honest_curve": np.cumprod(1 + h_ret)}


# ---------------------------------------------------------------------------------------------
# By hand: twelve days on a page
# ---------------------------------------------------------------------------------------------
HAND_SEED = 183   # two trades, one win, one loss, gaps between them (pick_hand_seed documents the search)


def hand(seed=HAND_SEED, n=12):
    o, c = bars(n, seed, start=50.0)
    o, c = np.round(o, 2), np.round(c, 2)
    sig, ret = two_down_rule(o, c)
    return {"open": o, "close": c, "signal": sig, "ret": ret, **summary(sig, ret)}


def pick_hand_seed(n=12, want=(3, 3)):
    """First seed whose page has 3 trades (enough to tally, few enough to read) and at least one loser."""
    for s in range(200):
        h = hand(s, n)
        r = h["ret"][h["signal"]]
        if want[0] <= h["trades"] <= want[1] and (r < 0).any() and (r > 0).any():
            return s
    raise RuntimeError


# ---------------------------------------------------------------------------------------------
# On a chart: the same rule, two years
# ---------------------------------------------------------------------------------------------
def chart(seed=5):
    o, c = bars(500, seed)
    sig, ret = two_down_rule(o, c)
    return {"open": o, "close": c, "signal": sig, "curve": np.cumprod(1 + ret), **summary(sig, ret)}


# ---------------------------------------------------------------------------------------------
# Not a prediction: the same rule, first five years vs next five, in 1,000 made-up markets
# ---------------------------------------------------------------------------------------------
def halves(worlds=1000, years=10, seed0=100):
    n = 252 * years
    first, second = [], []
    for w in range(worlds):
        o, c = bars(n, seed0 + w)
        sig, ret = two_down_rule(o, c)
        lr = np.log1p(ret)
        first.append(lr[: n // 2].sum())
        second.append(lr[n // 2:].sum())
    first, second = np.array(first), np.array(second)
    return {"worlds": worlds, "corr": float(np.corrcoef(first, second)[0, 1]),
            "first": first, "second": second,
            "first_up": float((first > 0).mean()),
            "good_first_then_down": float((second[first > np.quantile(first, 0.9)] < 0).mean())}

## By hand, twelve days
The paper moves one row at a time. On a trade day it moves in two steps: the open first, the close after.

In [ ]:
P = hand()
for i, (o, c) in enumerate(zip(P['open'], P['close'])):
    tag = f"TRADE {P['ret'][i]:+.2%}" if P['signal'][i] else ''
    print(f"day {i + 1:2d}  open {o:6.2f}  close {c:6.2f}  {'down' if c < o else 'up  '}  {tag}")
print(f"{P['trades']} trades, {P['wins']} winner, net {P['total']:+.2%}")

## The same loop, written the way you did it by hand
A backtest is this loop: at each step, decide only with what is above the paper, then move the paper.

In [ ]:
def by_hand(o, c):
    results, down_run = [], 0
    buy_next = False
    for day in range(len(o)):
        if buy_next:                     # the paper moves half a step: the open shows, so we buy it
            results.append(c[day] / o[day] - 1)   # then the close shows
        down_run = down_run + 1 if c[day] < o[day] else 0   # the whole day is above the paper now
        buy_next = down_run >= 2         # decision in pen, before the paper moves again
    return results

r = by_hand(P['open'], P['close'])
print([f'{x:+.2%}' for x in r])

## The hook: a rule that reads under the paper

In [ ]:
h = hook()
print('reads the close first:', h['peek'], '\n$100 to', round(100 * h['peek_curve'][-1]))
print('decides with yesterday:', h['honest'])

## Two years on a chart

In [ ]:
ch = chart()
print({k: ch[k] for k in ('trades', 'wins', 'win_share', 'total')})

## A test, not a prediction
One thousand made-up markets, ten years each: the first five years' result against the next five's.

In [ ]:
hv = halves()
print(f"correlation between halves: {hv['corr']:+.3f} over {hv['worlds']} markets")